# Treinamento de Modelos Deep Learning Darts

Foco em redes neurais avançadas para séries temporais.

### Importação das Bibliotecas
Framework principal: `darts` para a modelagem via PyTorch.

In [1]:
import pandas as pd
import numpy as np
import warnings
import os
import joblib
import logging
import torch
from sklearn.model_selection import TimeSeriesSplit
from darts import TimeSeries
from darts.models import BlockRNNModel, TCNModel
from darts.dataprocessing.transformers import Scaler

warnings.filterwarnings('ignore')
logging.getLogger("pytorch_lightning.utilities.rank_zero").setLevel(logging.WARNING)
logging.getLogger("pytorch_lightning.accelerators.cuda").setLevel(logging.WARNING)
os.makedirs('models', exist_ok=True)

In [2]:
def rmse(y_true, y_pred):
    y_true = np.array(y_true, dtype=float).flatten()
    y_pred = np.array(y_pred, dtype=float).flatten()
    return np.sqrt(np.mean((y_true - y_pred)**2))

def evaluate(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    r = rmse(y_true, y_pred)
    return {'MAE': round(mae, 2), 'RMSE': round(r, 2)}

### Preparação no Padrão Darts
Diferente dos modelos ML, aqui retornamos o DataFrame completo estruturado para ser convertido no formato nativo `TimeSeries` no momento do treino.

In [3]:
def load_data_darts(csv_path):
    df = pd.read_csv(csv_path, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
    target = 'Revenue'
    drop_cols = ['Date', target]
    feature_cols = [c for c in df.columns if c not in drop_cols and pd.api.types.is_numeric_dtype(df[c])]
    
    df[feature_cols] = df[feature_cols].ffill().bfill().fillna(0)
    return df, feature_cols

## 2. Carregamento dos Dados

### Carregamento das Partições

In [4]:
datasets = {
    'Produto': 'data/product.csv',
    'País': 'data/country.csv',
    'Cliente': 'data/customer.csv'
}

data = {}
for name, path in datasets.items():
    df, feat_cols = load_data_darts(path)
    data[name] = {'df': df, 'feature_cols': feat_cols}
    print(f"  {name}: total_samples={len(df)}")

  Produto: total_samples=194
  País: total_samples=268
  Cliente: total_samples=374


## 3. RNN (LSTM) via Darts (10-Fold Time Series Split)

### Estruturação e Escalonamento do BlockRNN (LSTM)
Em cada fold temporal, aplica-se o `Scaler` para deixar as features na mesma magnitude. 
Crucial: As covariáveis (`past_covariates`) do treino e validação são concatenadas (`cov_full_scaled`) para alimentar as etapas de projeção no futuro.

In [ ]:
N_SPLITS = 10
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
results_lstm = {}

pl_trainer_kwargs = {"enable_progress_bar": False}

for name, d in data.items():
    print(f"\nTreinando LSTM para a série: {name}")
    df = d['df']
    feature_cols = d['feature_cols']
    
    fold_metrics = []
    for fold, (train_idx, val_idx) in enumerate(tscv.split(df)):
        train_df = df.iloc[train_idx]
        val_df = df.iloc[val_idx]
        
        tgt_train = TimeSeries.from_dataframe(train_df, 'Date', 'Revenue')
        cov_train = TimeSeries.from_dataframe(train_df, 'Date', feature_cols)
        
        tgt_val = TimeSeries.from_dataframe(val_df, 'Date', 'Revenue')
        cov_val = TimeSeries.from_dataframe(val_df, 'Date', feature_cols)
        
        scaler_tgt = Scaler()
        scaler_cov = Scaler()
        
        tgt_train_scaled = scaler_tgt.fit_transform(tgt_train)
        cov_train_scaled = scaler_cov.fit_transform(cov_train)
        cov_val_scaled = scaler_cov.transform(cov_val)
        cov_full_scaled = cov_train_scaled.append(cov_val_scaled)
        
        model = BlockRNNModel(
            model='LSTM',
            hidden_dim=20,
            n_rnn_layers=1,
            dropout=0.1,
            batch_size=16,
            n_epochs=20,
            optimizer_kwargs={'lr': 1e-3},
            input_chunk_length=7,
            output_chunk_length=1,
            random_state=42,
            pl_trainer_kwargs=pl_trainer_kwargs
        )
        
        if len(tgt_train_scaled) > 7:
            model.fit(tgt_train_scaled, past_covariates=cov_train_scaled)
            pred_scaled = model.predict(n=len(tgt_val), series=tgt_train_scaled, past_covariates=cov_full_scaled)
            pred = scaler_tgt.inverse_transform(pred_scaled)
            
            y_pred = np.maximum(pred.values(), 0)
            y_val = tgt_val.values()
            metrics = evaluate(y_val, y_pred)
            fold_metrics.append(metrics)
            
            if fold == N_SPLITS - 1:
                model.save(f'models/result_pkl/lstm_{name}.pt')
                
    if fold_metrics:
        avg_mae = np.mean([m['MAE'] for m in fold_metrics])
        avg_rmse = np.mean([m['RMSE'] for m in fold_metrics])
        results_lstm[name] = {'MAE': avg_mae, 'RMSE': avg_rmse}
        print(f"  {name} LSTM Médio (10 Folds): MAE={avg_mae:.2f}, RMSE={avg_rmse:.2f}")


Treinando LSTM para a série: Produto



  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | rnn             | LSTM             | 4.2 K  | train
6 | fc              | Sequential       | 21     | train
-------------------------------------------------------------
4.3 K     Trainable params
0         Non-trainable params
4.3 K     Total params
0.017     Total estimated model params size (MB)
8         Modules in train mode
0         Modules in eval mode
`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relativ

  Produto LSTM Médio (10 Folds): MAE=108.11, RMSE=130.08

Treinando LSTM para a série: País



  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | rnn             | LSTM             | 4.2 K  | train
6 | fc              | Sequential       | 21     | train
-------------------------------------------------------------
4.2 K     Trainable params
0         Non-trainable params
4.2 K     Total params
0.017     Total estimated model params size (MB)
8         Modules in train mode
0         Modules in eval mode
`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relativ

  País LSTM Médio (10 Folds): MAE=4191.12, RMSE=5209.97

Treinando LSTM para a série: Cliente


`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.

  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | rnn             | LSTM             | 4.2 K  | train
6 | fc              | Sequential       | 21     | train
-------------------------------------------------------------
4.2 K     Trainable params
0         Non-trainable params
4.2 K     Total params
0.017     Total estimated 

  Cliente LSTM Médio (10 Folds): MAE=395.94, RMSE=593.86


## 4. TCN via Darts (10-Fold Time Series Split)


### Treinamento de Redes Convolucionais Temporais (TCN)
Baseando-se em convoluções causais dilatadas, avalia janelas de tempo completas com alto poder de extração de picos locais.

In [ ]:
results_tcn = {}

for name, d in data.items():
    print(f"\nTreinando TCN para a série: {name}")
    df = d['df']
    feature_cols = d['feature_cols']
    
    fold_metrics = []
    for fold, (train_idx, val_idx) in enumerate(tscv.split(df)):
        train_df = df.iloc[train_idx]
        val_df = df.iloc[val_idx]
        
        tgt_train = TimeSeries.from_dataframe(train_df, 'Date', 'Revenue')
        cov_train = TimeSeries.from_dataframe(train_df, 'Date', feature_cols)
        
        tgt_val = TimeSeries.from_dataframe(val_df, 'Date', 'Revenue')
        cov_val = TimeSeries.from_dataframe(val_df, 'Date', feature_cols)
        
        scaler_tgt = Scaler()
        scaler_cov = Scaler()
        
        tgt_train_scaled = scaler_tgt.fit_transform(tgt_train)
        cov_train_scaled = scaler_cov.fit_transform(cov_train)
        cov_val_scaled = scaler_cov.transform(cov_val)
        cov_full_scaled = cov_train_scaled.append(cov_val_scaled)
        
        model = TCNModel(
            input_chunk_length=7,
            output_chunk_length=1,
            dropout=0.1,
            batch_size=16,
            n_epochs=20,
            optimizer_kwargs={'lr': 1e-3},
            random_state=42,
            pl_trainer_kwargs=pl_trainer_kwargs
        )
        
        if len(tgt_train_scaled) > 7:
            model.fit(tgt_train_scaled, past_covariates=cov_train_scaled)
            pred_scaled = model.predict(n=len(tgt_val), series=tgt_train_scaled, past_covariates=cov_full_scaled)
            pred = scaler_tgt.inverse_transform(pred_scaled)
            
            y_pred = np.maximum(pred.values(), 0)
            y_val = tgt_val.values()
            metrics = evaluate(y_val, y_pred)
            fold_metrics.append(metrics)
            
            if fold == N_SPLITS - 1:
                model.save(f'models/result_pkl/tcn_{name}.pt')
                
    if fold_metrics:
        avg_mae = np.mean([m['MAE'] for m in fold_metrics])
        avg_rmse = np.mean([m['RMSE'] for m in fold_metrics])
        results_tcn[name] = {'MAE': avg_mae, 'RMSE': avg_rmse}
        print(f"  {name} TCN Médio (10 Folds): MAE={avg_mae:.2f}, RMSE={avg_rmse:.2f}")



  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | res_blocks      | ModuleList       | 452    | train
-------------------------------------------------------------
452       Trainable params
0         Non-trainable params
452       Total params
0.002     Total estimated model params size (MB)
18        Modules in train mode
0         Modules in eval mode



Treinando TCN para a série: Produto


`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.

  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | res_blocks      | ModuleList       | 452    | train
-------------------------------------------------------------
452       Trainable params
0         Non-trainable params
452       Total params
0.002     Total estimated model params size (MB)
18        Modules in train mode
0

  Produto TCN Médio (10 Folds): MAE=379.54, RMSE=564.60

Treinando TCN para a série: País


`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.

  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | res_blocks      | ModuleList       | 440    | train
-------------------------------------------------------------
440       Trainable params
0         Non-trainable params
440       Total params
0.002     Total estimated model params size (MB)
18        Modules in train mode
0

  País TCN Médio (10 Folds): MAE=96136.46, RMSE=351800.34

Treinando TCN para a série: Cliente



  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | res_blocks      | ModuleList       | 440    | train
-------------------------------------------------------------
440       Trainable params
0         Non-trainable params
440       Total params
0.002     Total estimated model params size (MB)
18        Modules in train mode
0         Modules in eval mode
`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warnin

  Cliente TCN Médio (10 Folds): MAE=603.17, RMSE=878.22


### Exportação dos Resultados
Métricas médias persistidas para as duas abordagens neurais.

In [7]:
import json
with open('models/result_json/resultados_dl.json', 'w') as f:
    json.dump({'LSTM': results_lstm, 'TCN': results_tcn}, f, indent=4)
print("Resultados salvos em models/result_json/resultados_dl.json")


Resultados salvos em models/result_json/resultados_dl.json
